## Deploy fine-tuned Amazon Nova Model with on-demand inference

In [ ]:
%pip install --quiet -U boto3 botocore

In [ ]:
import boto3 
from botocore.config import Config
import ipywidgets as widgets
from IPython.display import display
import time 
from utils.bedrock import get_fine_tunable_vision_models, wait_for, deployment_status

### Regions that support custom model for on-demand inference

As of October 2025 Amazon Bedrock supports on-demand inference with custom models in the following regions:
* us-east-1
* us-west-2

Please review the Amazon Bedrock Documentation for the latest information: [Amazon Bedrock User Guide - Deploy a custom model for on-demand inference](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-custom-model-on-demand.html)

In [ ]:
on_demand_inference_supported_region = ["us-east-1", "us-west-2"]

In [ ]:
default_region = "us-east-1"

In [ ]:
region_dropdown = widgets.Dropdown(
    options=on_demand_inference_supported_region,
    value=default_region,
    description='Region:',
    style={'description_width': 'initial'},
    layout={'width': 'auto'}
)


# Create output widget for status messages
output = widgets.Output()

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select AWS Region</h3>"),
    region_dropdown,
    output
]))


In [ ]:
region=region_dropdown.value
boto_session = boto3.Session(
    region_name=region
)

In [ ]:
my_config = Config(
    region_name = region, 
    signature_version = 'v4',
    retries = {
        'max_attempts': 5,
        'mode': 'standard'
    })

bedrock = boto_session.client(service_name="bedrock", config=my_config)

In [ ]:
fine_tunable_models = get_fine_tunable_vision_models(bedrock)

### Supported base models for on-demand inference of fine-tuned models

As of October 2025 Amazon Bedrock supports on-demand inference with custom models for the following models:
* Amazon Nova Lite
* Amazon Nova Micro
* Amazon Nova Pro
* Meta Llama 3.3 70B Instruct

Keep in mind that Amazon Nova Micro and Meta Llama 3.3 70B Instruct do not support images as input so they are irrelevant for out document processing use case.

Please review the Amazon Bedrock Documentation for the latest information: [Amazon Bedrock User Guide - Deploy a custom model for on-demand inference](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-custom-model-on-demand.html)

In [ ]:
on_demand_inference_supported_based_models = [
    "amazon.nova-pro-v1:0",
    "amazon.nova-lite-v1:0",
    "amazon.nova-micro-v1:0",
    "meta.llama3-3-70b-instruct-v1:0",
    "amazon.nova-2-lite-v1:0"
]

In [ ]:
on_demand_models = [model["modelArn"] for model in fine_tunable_models if any(od_model_id in model["modelId"] for od_model_id in on_demand_inference_supported_based_models)]

Let's retrieve your fine-tuned custom models:

In [ ]:
response = bedrock.list_model_customization_jobs(
    statusEquals='Completed',
    sortBy='CreationTime',
    sortOrder='Descending'
)

Filter the custom models to only contain the ones that support on-demand inference:

In [ ]:
custom_models = [custom_model for custom_model in response['modelCustomizationJobSummaries'] if custom_model['baseModelArn'] in on_demand_models]

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import json

jobs = custom_models

# Create dropdown
model_dropdown = widgets.Dropdown(
    options=[(f"{job['jobName']} - {job['status']}", idx) 
             for idx, job in enumerate(jobs)],
    description='Select Job:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)



def display_job_details(change):
    with output:
        output.clear_output()
        job_idx = change['new']
        job = jobs[job_idx]
        
        # Create nice HTML display
        html = f"""
        <div style="padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
            <h3>{job['jobName']}</h3>
            <table style="width: 100%; border-collapse: collapse;">
                <tr><td style="padding: 5px;"><b>Status:</b></td><td>{job['status']}</td></tr>
                <tr><td style="padding: 5px;"><b>End Time:</b></td><td>{job['endTime'].strftime('%Y-%m-%d %H:%M')}</td></tr>
                <tr><td style="padding: 5px;"><b>Custom Model Name:</b></td><td>{job.get('customModelName', 'N/A')}</td></tr>
                <tr><td style="padding: 5px;"><b>Customization Type:</b></td><td>{job.get('customizationType', 'N/A')}</td></tr>
                <tr><td style="padding: 5px;"><b>Job ARN:</b></td><td style="word-break: break-all;">{job['jobArn']}</td></tr>
                <tr><td style="padding: 5px;"><b>Base Model ARN:</b></td><td style="word-break: break-all;">{job['baseModelArn']}</td></tr>
            </table>
        </div>
        """
        display(HTML(html))
    

model_dropdown.observe(display_job_details, names='value')
display(model_dropdown, output)

# Trigger initial display
if jobs:
    display_job_details({'new': 0})


In [ ]:
# Function to create on-demand inferencing deployment for custom model
def create_model_deployment(custom_model_arn):
    """
    Create an on-demand inferencing deployment for the custom model
    
    Parameters:
    -----------
    custom_model_arn : str
        ARN of the custom model to deploy
        
    Returns:
    --------
    deployment_arn : str
        ARN of the created deployment
    """
    try:
        print(f"Creating on-demand inferencing deployment for model: {custom_model_arn}")
        
        # Generate a unique name for the deployment
        deployment_name = f"nova-ocr-deployment-{time.strftime('%Y%m%d-%H%M%S')}"
        
        # Create the deployment
        response = bedrock.create_custom_model_deployment(
            modelArn=custom_model_arn,
            modelDeploymentName=deployment_name,
            description=f"on-demand inferencing deployment for model: {custom_model_arn}",
        )
        
        # Get the deployment ARN
        deployment_arn = response.get('customModelDeploymentArn')
        
        print(f"Deployment request submitted. Deployment ARN: {deployment_arn}")
        return deployment_arn
    
    except Exception as e:
        print(f"Error creating deployment: {e}")
        return None

In [ ]:
print(f"Deploying: {jobs[model_dropdown.value]['customModelName']}")

In [ ]:
# Create on-demand deployment
deployment_arn = create_model_deployment(jobs[model_dropdown.value]['customModelArn'])

if deployment_arn:
    # Check initial status
    initial_status = deployment_status(deployment_arn, bedrock)

Wait for Deployment to Complete

ℹ️ Info: It takes about 30 mins to complete the deployment

Let's monitor our deployment until it's ready for use:

In [ ]:

# Only attempt to wait for deployment if it was created
if deployment_arn: 
    # wait for 1 hour for the training job to complete
    deployment_ready = wait_for(deployment_status, max_wait_seconds=3600, check_interval=60, deployment_arn=deployment_arn, bedrock=bedrock)
    if deployment_ready:
        print(f"Deployment ready. Model id: {deployment_arn}")
    else:
        print("Deployment did not complete successfully.")